# Distances Between Observations

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [37]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?

In [38]:
df_ames = pd.read_csv("https://dlsun.github.io/pods/data/AmesHousing.txt", sep = "\t")

df_ames.shape

(2930, 82)

In [39]:
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 *df_ames["Half Bath"]

features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms",
            "House Style", "Neighborhood", "Year Built", "SalePrice"]

distance_features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Year Built", "SalePrice"]
df_ames.loc[[0], distance_features]

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Year Built,SalePrice
0,1656,3,1.0,1960,215000


In [40]:
df_cheaper = df_ames[df_ames["SalePrice"] < df_ames.loc[0, "SalePrice"]].copy()
df_cheaper["distance"] = (((df_cheaper[distance_features] - df_ames.loc[0, distance_features]) ** 2).sum(axis=1)) ** 0.5
df_cheaper.sort_values("distance")[features + ["distance"]]


,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,distance
319,1894,4,2.5,2Story,Timber,2002,214900,261.555443
747,2134,5,2.0,2.5Unf,BrkSide,1916,214500,693.126972
2775,1588,3,2.0,1Story,CollgCr,1999,214000,1003.068293
2791,1755,3,2.5,2Story,CollgCr,1999,214000,1005.646185
1472,1536,3,2.0,1Story,CollgCr,2002,214000,1008.050098
...,...,...,...,...,...,...,...,...
2880,480,1,0.0,1Story,IDOTRR,1949,35311,179692.848558
2843,498,1,1.0,1Story,Edwards,1922,35000,180003.728884
726,720,2,1.0,1Story,IDOTRR,1920,34900,180102.436677
1553,733,2,1.0,1Story,IDOTRR,1952,13100,201902.109930


In [41]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style"]

X = pd.get_dummies(df_ames[features], columns=["House Style"], dtype=int)

cols = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]

X_scaled = X.copy()
X_scaled[cols] = (X_scaled[cols] - X_scaled[cols].mean()) / X_scaled[cols].std()

df_cheaper["euclidean"] = (((X_scaled.loc[df_cheaper.index] - X_scaled.loc[0]) ** 2).sum(axis=1)) ** 0.5

df_cheaper["manhattan"] = abs(X_scaled.loc[df_cheaper.index] - X_scaled.loc[0]).sum(axis=1)

df_cheaper.sort_values("euclidean").head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Bathrooms,distance,euclidean,manhattan
1940,1941,535353050,20,RL,75.0,9532,Pave,NaN,Reg,Lvl,...,0,2,2007,WD,Normal,153000,1.0,62000.001048,0.017804,0.017804
618,619,534476150,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,...,0,10,2009,WD,Normal,167000,1.0,48000.002010,0.023738,0.023738
2700,2701,904100170,20,RL,100.0,21370,Pave,NaN,Reg,Lvl,...,600,6,2006,WD,Normal,131000,1.0,84000.002119,0.031651,0.031651
314,315,916125360,20,RL,NaN,57200,Pave,NaN,IR1,Bnk,...,0,6,2010,WD,Normal,160000,1.0,55000.010045,0.061324,0.061324
788,789,905450020,20,RL,73.0,9855,Pave,NaN,Reg,Lvl,...,0,11,2009,COD,Normal,127500,1.0,87500.006314,0.065281,0.065281


In [42]:
df_cheaper.sort_values("manhattan").head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Bathrooms,distance,euclidean,manhattan
1940,1941,535353050,20,RL,75.0,9532,Pave,NaN,Reg,Lvl,...,0,2,2007,WD,Normal,153000,1.0,62000.001048,0.017804,0.017804
618,619,534476150,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,...,0,10,2009,WD,Normal,167000,1.0,48000.002010,0.023738,0.023738
2700,2701,904100170,20,RL,100.0,21370,Pave,NaN,Reg,Lvl,...,600,6,2006,WD,Normal,131000,1.0,84000.002119,0.031651,0.031651
314,315,916125360,20,RL,NaN,57200,Pave,NaN,IR1,Bnk,...,0,6,2010,WD,Normal,160000,1.0,55000.010045,0.061324,0.061324
788,789,905450020,20,RL,73.0,9855,Pave,NaN,Reg,Lvl,...,0,11,2009,COD,Normal,127500,1.0,87500.006314,0.065281,0.065281


Based on the results, house 319 is the closest cheaper match to house 0 with a distance of 261.56. It has 1,894 square feet, 4 bedrooms, 2.5 bathrooms, and was built in 2002. Its sale price was $214,900. The other matches have much larger distances, so house 319 appears to be the strongest similar cheaper option.

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [43]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style"]

X = pd.get_dummies(df_ames[features], columns=["House Style"], dtype=int)

X.head()


df_cheaper = df_ames[df_ames["SalePrice"] < df_ames.loc[0, "SalePrice"]].copy()

df_cheaper["distance"] = (((X.loc[df_cheaper.index] - X.loc[0]) ** 2).sum(axis=1)) ** 0.5

df_cheaper.sort_values("distance").head()[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms",
                                          "House Style", "Neighborhood", "Year Built",
                                          "SalePrice", "distance"]]



,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,distance
1927,1657,3,2.0,1Story,NAmes,1970,163500,1.414214
1197,1656,4,2.0,1Story,NWAmes,1973,135000,1.414214
1550,1656,3,1.5,SLvl,IDOTRR,1967,126000,1.500000
2638,1657,4,1.0,1.5Fin,OldTown,1920,111500,2.000000
1293,1656,2,2.0,1.5Fin,OldTown,1940,119164,2.000000


In [44]:
df_cheaper["manhattan"] = abs(X.loc[df_cheaper.index] - X.loc[0]).sum(axis=1)

cols = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]
X_scaled = X.copy()
X_scaled[cols] = (X_scaled[cols] - X_scaled[cols].mean()) / X_scaled[cols].std()

df_cheaper["euclidean_scaled"] = (((X_scaled.loc[df_cheaper.index] - X_scaled.loc[0]) ** 2).sum(axis=1)) ** 0.5
df_cheaper["manhattan_scaled"] = abs(X_scaled.loc[df_cheaper.index] - X_scaled.loc[0]).sum(axis=1)

In [45]:
df_cheaper.sort_values("manhattan").head()
df_cheaper.sort_values("euclidean_scaled").head()
df_cheaper.sort_values("manhattan_scaled").head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Bathrooms,distance,manhattan,euclidean_scaled,manhattan_scaled
1940,1941,535353050,20,RL,75.0,9532,Pave,NaN,Reg,Lvl,...,2,2007,WD,Normal,153000,1.0,9.0,9.0,0.017804,0.017804
618,619,534476150,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,...,10,2009,WD,Normal,167000,1.0,12.0,12.0,0.023738,0.023738
2700,2701,904100170,20,RL,100.0,21370,Pave,NaN,Reg,Lvl,...,6,2006,WD,Normal,131000,1.0,16.0,16.0,0.031651,0.031651
314,315,916125360,20,RL,NaN,57200,Pave,NaN,IR1,Bnk,...,6,2010,WD,Normal,160000,1.0,31.0,31.0,0.061324,0.061324
788,789,905450020,20,RL,73.0,9855,Pave,NaN,Reg,Lvl,...,11,2009,COD,Normal,127500,1.0,33.0,33.0,0.065281,0.065281


The closest cheaper homes are generally similar to house 0 in living area, bedrooms, bathrooms, and house style. The rankings change somewhat depending on whether the variables are scaled and whether Euclidean or Manhattan distance is used, so the results are somewhat sensitive to those choices. Scaling is important because living area is measured on a much larger scale than bedrooms, bathrooms, and the dummy variables for house style.

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [46]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms",
            "Year Built", "House Style", "Neighborhood"]

X = pd.get_dummies(df_ames[features],
                   columns=["House Style", "Neighborhood"],
                   dtype=int)

cols = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Year Built"]
X[cols] = (X[cols] - X[cols].mean()) / X[cols].std()

df_cheaper = df_ames[df_ames["SalePrice"] < df_ames.loc[0, "SalePrice"]].copy()

df_cheaper["euclidean"] = (((X.loc[df_cheaper.index] - X.loc[0]) ** 2).sum(axis=1)) ** 0.5

df_cheaper["manhattan"] = abs(X.loc[df_cheaper.index] - X.loc[0]).sum(axis=1)

df_cheaper.sort_values("euclidean").head()[[
    "Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Year Built",
    "House Style", "Neighborhood", "SalePrice",
    "euclidean", "manhattan"
]]

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Year Built,House Style,Neighborhood,SalePrice,euclidean,manhattan
1240,1570,3,1.0,1958,1Story,NAmes,166800,0.182525,0.236251
1940,1647,3,1.0,1953,1Story,NAmes,153000,0.232124,0.249244
618,1644,3,1.0,1953,1Story,NAmes,167000,0.232655,0.255179
2558,1433,3,1.0,1961,1Story,NAmes,161000,0.442377,0.474203
1896,1429,3,1.0,1960,1Story,NAmes,181900,0.449052,0.449052


The closest cheaper homes are very similar to house 0 because they have nearly the same living area, bedroom count, bathroom count, house style, neighborhood, and year built.

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [47]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [48]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

City                                                        San Luis Obispo
State                                                                    CA
AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
PCIP01                                                               0.1084
PCIP03                                                               0.0255
PCIP04                                                               0.0441
PCIP05                                                               0.0019
PCIP09                                                               0.0353
PCIP10                                                               0.0175
PCIP11                                                               0.0326
PCIP12      

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [49]:
df_college.columns

Index(['City', 'State', 'AdmissionRate', 'Undergraduates',
       'CarnegieClassification', 'Ownership', 'PCIP01', 'PCIP03', 'PCIP04',
       'PCIP05', 'PCIP09', 'PCIP10', 'PCIP11', 'PCIP12', 'PCIP13', 'PCIP14',
       'PCIP15', 'PCIP16', 'PCIP19', 'PCIP22', 'PCIP23', 'PCIP24', 'PCIP25',
       'PCIP26', 'PCIP27', 'PCIP29', 'PCIP30', 'PCIP31', 'PCIP38', 'PCIP39',
       'PCIP40', 'PCIP41', 'PCIP42', 'PCIP43', 'PCIP44', 'PCIP45', 'PCIP46',
       'PCIP47', 'PCIP48', 'PCIP49', 'PCIP50', 'PCIP51', 'PCIP52', 'PCIP54'],
      dtype='str')

In [50]:
features = ["AdmissionRate", "Undergraduates"]

X = df_college[features].copy()
X = (X - X.mean()) / X.std()

df_college["distance"] = (((X - X.loc[school_name]) ** 2).sum(axis=1)) ** 0.5

df_college.sort_values("distance").head(10)[features + ["distance"]]

,AdmissionRate,Undergraduates,distance
Institution,,,
California Polytechnic State University-San Luis Obispo,0.3300,21090.0,0.000000
University of California-Santa Barbara,0.2918,23081.0,0.309162
DeVry University-Illinois,0.4552,19729.0,0.593121
University of North Carolina at Chapel Hill,0.2040,19722.0,0.596846
Clemson University,0.4922,21577.0,0.736788
University of Virginia-Main Campus,0.2074,17041.0,0.761296
CUNY Hunter College,0.4590,17293.0,0.761441
Stony Brook University,0.4806,17900.0,0.795756
Boston University,0.1865,17501.0,0.797041


The most similar schools to Cal Poly are those with admission rates and undergraduate populations closest to Cal Poly, with the University of Central Florida appearing to be the closest match based on the smallest standardized Euclidean distance.

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [51]:
features = ["AdmissionRate", "Undergraduates",
            "CarnegieClassification", "Ownership"]

X = pd.get_dummies(df_college[features],
                   columns=["CarnegieClassification", "Ownership"],
                   dtype=int)

cols = ["AdmissionRate", "Undergraduates"]
X[cols] = (X[cols] - X[cols].mean()) / X[cols].std()

df_college["distance2"] = (((X - X.loc[school_name]) ** 2).sum(axis=1)) ** 0.5

df_college.sort_values("distance2").head(10)[features + ["distance2"]]

,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,distance2
Institution,,,,,
California Polytechnic State University-San Luis Obispo,0.3300,21090.0,Master's Colleges & Universities: Larger Programs,Public,0.000000
CUNY Hunter College,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761441
CUNY Bernard M Baruch College,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public,1.073601
CUNY John Jay College of Criminal Justice,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public,1.184989
CUNY Brooklyn College,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public,1.376321
University of California-Santa Barbara,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,1.447612
California State Polytechnic University-Pomona,0.6062,26802.0,Master's Colleges & Universities: Larger Programs,Public,1.450297
CUNY Queens College,0.6078,14859.0,Master's Colleges & Universities: Larger Programs,Public,1.491386
DeVry University-Illinois,0.4552,19729.0,Master's Colleges & Universities: Larger Programs,Private for-profit,1.533555


The closest schools are mostly public Master’s Colleges with admission rates and undergraduate enrollments similar to Cal Poly, which makes sense because the categorical variables now push schools with the same Carnegie classification and ownership closer in the ranking.

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [52]:
pcip_cols = [col for col in df_college.columns if col.startswith("PCIP")]

X = df_college[pcip_cols].copy()

df_college["pcip_distance"] = (((X - X.loc[school_name]) ** 2).sum(axis=1)) ** 0.5

df_college.sort_values("pcip_distance").head(10)[pcip_cols + ["pcip_distance"]]

,PCIP01,PCIP03,PCIP04,PCIP05,PCIP09,PCIP10,PCIP11,PCIP12,PCIP13,PCIP14,...,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54,pcip_distance
Institution,,,,,,,,,,,,,,,,,,,,,
California Polytechnic State University-San Luis Obispo,0.1084,0.0255,0.0441,0.0019,0.0353,0.0175,0.0326,0.0000,0.0130,0.2314,...,0.0588,0.0,0.0,0.0,0.0000,0.0130,0.0060,0.1637,0.0110,0.000000
North Carolina State University at Raleigh,0.0906,0.0356,0.0075,0.0000,0.0353,0.0000,0.0366,0.0000,0.0233,0.2417,...,0.0467,0.0,0.0,0.0,0.0000,0.0155,0.0000,0.1580,0.0116,0.084343
Iowa State University,0.1013,0.0131,0.0163,0.0009,0.0392,0.0000,0.0428,0.0000,0.0473,0.2421,...,0.0203,0.0,0.0,0.0,0.0000,0.0399,0.0131,0.1651,0.0069,0.085054
University of Illinois Urbana-Champaign,0.0625,0.0072,0.0121,0.0029,0.0633,0.0000,0.0441,0.0000,0.0227,0.2131,...,0.0776,0.0,0.0,0.0,0.0000,0.0263,0.0416,0.1177,0.0100,0.111475
Mississippi State University,0.0619,0.0263,0.0208,0.0000,0.0360,0.0000,0.0201,0.0002,0.0712,0.1717,...,0.0367,0.0,0.0,0.0,0.0000,0.0102,0.0040,0.1825,0.0069,0.118799
Texas A & M University-College Station,0.0847,0.0241,0.0111,0.0001,0.0397,0.0000,0.0334,0.0000,0.0272,0.1627,...,0.0593,0.0,0.0,0.0,0.0048,0.0010,0.0582,0.1593,0.0094,0.123973
Clemson University,0.0628,0.0115,0.0185,0.0029,0.0049,0.0000,0.0332,0.0000,0.0336,0.1976,...,0.0700,0.0,0.0,0.0,0.0000,0.0286,0.0766,0.1840,0.0080,0.128333
Purdue University-Main Campus,0.0604,0.0111,0.0016,0.0017,0.0167,0.0000,0.0963,0.0000,0.0316,0.2252,...,0.0447,0.0,0.0,0.0,0.0135,0.0160,0.0538,0.1698,0.0047,0.134177
Virginia Polytechnic Institute and State University,0.0436,0.0319,0.0178,0.0000,0.0343,0.0000,0.0490,0.0000,0.0000,0.2221,...,0.0867,0.0,0.0,0.0,0.0000,0.0222,0.0000,0.2051,0.0093,0.137570


The most similar schools are the ones with the smallest Euclidean distance across the PCIP proportions, meaning their students are distributed across fields of study most similarly to Cal Poly.